# Inside the Vocal Tract: LPC Analysis

Linear Predictive Coding models speech as a filter (the vocal tract) driven by an excitation source (the glottis). The AR coefficients capture the resonances (formants) of your mouth and throat shape.

This notebook demonstrates LPC estimation, spectral envelopes, coefficient representations, and bandwidth expansion.

In [ ]:
# Install dependencies if needed (e.g. on Colab)
# %pip install py-voicebox

In [ ]:
import os
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display
from pyvoicebox import v_lpcauto, v_lpcar2cc, v_lpcar2rf, v_lpcar2ls, v_lpcbwexp
from pyvoicebox import v_enframe, v_windows

# Download
url = "http://festvox.org/cmu_arctic/cmu_arctic/cmu_us_bdl_arctic/wav/arctic_a0003.wav"
wav_path = "arctic_a0003.wav"
if not os.path.exists(wav_path):
    print("Downloading...")
    urllib.request.urlretrieve(url, wav_path)

signal, fs = sf.read(wav_path)
print(f"Loaded: {fs} Hz, {len(signal)/fs:.2f}s")
display(Audio(signal, rate=fs))

## Extract a voiced frame

We pick the frame with the highest energy — this guarantees a strongly voiced vowel
with clear harmonic and formant structure.

In [ ]:
# Pick the frame with highest energy (guaranteed voiced)
frame_len = int(0.025 * fs)  # 25 ms
frame_hop = int(0.010 * fs)  # 10 ms
frames, _, _ = v_enframe(signal, frame_len, frame_hop)
energies = np.sum(frames**2, axis=1)
best_idx = np.argmax(energies)
frame = frames[best_idx]
win = v_windows(3, frame_len).flatten()  # Hamming

fig, ax = plt.subplots(figsize=(10, 2.5))
t_frame = np.arange(frame_len) / fs * 1000  # ms
ax.plot(t_frame, frame, color='#3f51b5', linewidth=1.2)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Amplitude')
ax.set_title(f'Highest-energy frame (25 ms, frame {best_idx})')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## LPC spectral envelope vs FFT

The FFT shows all the harmonic detail — individual peaks at multiples of F0. The LPC envelope traces the smooth formant peaks, ignoring the fine harmonic structure.

The peaks in the LPC envelope correspond to **formants** (F1, F2, F3...) — the resonant frequencies of the vocal tract. For a typical male vowel, expect F1 around 300-800 Hz, F2 around 800-2500 Hz.

In [ ]:
lpc_order = 16
ar, e, k = v_lpcauto(frame * win, lpc_order)

# FFT spectrum
n_fft = 1024
fft_spec = np.fft.rfft(frame * win, n_fft)
fft_db = 20 * np.log10(np.abs(fft_spec) + 1e-10)
fft_freqs = np.arange(len(fft_spec)) * fs / n_fft

# LPC spectrum
ar_flat = ar.flatten()
w = np.exp(-1j * 2 * np.pi * np.arange(n_fft // 2 + 1) / n_fft)
H = np.array([1.0 / np.polyval(ar_flat[::-1], wi) for wi in w])
lpc_db = 20 * np.log10(np.abs(H) + 1e-10)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fft_freqs, fft_db - np.max(fft_db), color='#90a4ae', linewidth=0.8,
        alpha=0.7, label='FFT spectrum')
ax.plot(fft_freqs, lpc_db - np.max(lpc_db), color='#e91e63', linewidth=2.5,
        label=f'LPC envelope (order {lpc_order})')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('LPC spectral envelope vs FFT spectrum')
ax.set_xlim(0, fs / 2)
ax.set_ylim(-60, 5)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Effect of LPC order

Higher order = more spectral detail. Too low and you miss formants;
too high and you start modelling individual harmonics instead of the envelope.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fft_freqs, fft_db - np.max(fft_db), color='#e0e0e0', linewidth=0.8)

colors = ['#2196f3', '#4caf50', '#ff9800', '#e91e63', '#9c27b0']
for order, color in zip([4, 8, 12, 16, 20], colors):
    ar_o, _, _ = v_lpcauto(frame * win, order)
    ar_o = ar_o.flatten()
    H_o = np.array([1.0 / np.polyval(ar_o[::-1], wi) for wi in w])
    db_o = 20 * np.log10(np.abs(H_o) + 1e-10)
    ax.plot(fft_freqs, db_o - np.max(db_o), color=color, linewidth=1.8,
            label=f'Order {order}')

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('LPC order comparison')
ax.set_xlim(0, fs / 2)
ax.set_ylim(-50, 5)
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Coefficient representations

The same vocal tract model can be expressed as AR coefficients, cepstral coefficients, reflection coefficients, or line spectral frequencies. Each has different properties:

- **AR coefficients** — direct filter form, but sensitive to quantisation
- **Cepstral coefficients** — decorrelated, good for distance metrics (used in MFCC-based recognition)
- **Reflection coefficients** — bounded in [-1, 1], guaranteed stable, used in lattice filters
- **Line spectral frequencies** — pairs of frequencies in [0, pi], good interpolation properties, preferred for speech coding

In [ ]:
cc = v_lpcar2cc(ar)[0]  # cepstral (returns tuple)
rf = v_lpcar2rf(ar)      # reflection (returns ndarray)
ls = v_lpcar2ls(ar)      # line spectral (returns ndarray)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].stem(ar.flatten(), linefmt='#3f51b5', markerfmt='o', basefmt='grey')
axes[0, 0].set_title('AR coefficients')
axes[0, 0].set_xlabel('Index')
axes[0, 0].grid(True, alpha=0.2)

axes[0, 1].stem(cc.flatten()[:20], linefmt='#e91e63', markerfmt='o', basefmt='grey')
axes[0, 1].set_title('Cepstral coefficients')
axes[0, 1].set_xlabel('Index')
axes[0, 1].grid(True, alpha=0.2)

axes[1, 0].stem(rf.flatten(), linefmt='#4caf50', markerfmt='o', basefmt='grey')
axes[1, 0].set_title('Reflection coefficients')
axes[1, 0].set_xlabel('Index')
axes[1, 0].set_ylim(-1, 1)
axes[1, 0].grid(True, alpha=0.2)

axes[1, 1].stem(ls.flatten(), linefmt='#ff9800', markerfmt='o', basefmt='grey')
axes[1, 1].set_title('Line spectral frequencies')
axes[1, 1].set_xlabel('Index')
axes[1, 1].set_ylabel('Frequency (rad)')
axes[1, 1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## Bandwidth expansion

`v_lpcbwexp` widens the formant peaks by pulling the LPC poles toward the origin.
This is used in speech synthesis and coding to control spectral smoothness.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Compute all curves, use absolute dB (not normalised)
for gamma, color, label in [
    (1.0, '#e91e63', 'Original (\u03b3=1.0)'),
    (0.98, '#ff9800', '\u03b3=0.98'),
    (0.94, '#4caf50', '\u03b3=0.94'),
    (0.88, '#2196f3', '\u03b3=0.88'),
]:
    ar_exp = v_lpcbwexp(ar, gamma).flatten()
    H_g = np.array([1.0 / np.polyval(ar_exp[::-1], wi) for wi in w])
    db_g = 20 * np.log10(np.abs(H_g) + 1e-10)
    ax.plot(fft_freqs, db_g, color=color, linewidth=2, label=label)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title('Bandwidth expansion: smoothing the spectral envelope')
ax.set_xlim(0, fs / 2)
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()